In [ ]:
#nomor 1
import multiprocessing as mp
import time
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Definisi Fungsi Eksperimen (Sesuai Eksperimen 7.1) ---
def is_prime(n):
    if n < 2: 
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0: 
            return False
    return True

def worker_task(numbers_chunk):
    # Menghitung jumlah bilangan prima dalam satu chunk/bagian data
    return sum(1 for num in numbers_chunk if is_prime(num))

def run_experiment(num_workers, data):
    # Membagi data menjadi chunk sesuai jumlah worker
    chunks = np.array_split(data, num_workers)
    
    # Menggunakan konteks 'fork' sesuai instruksi soal
    ctx = mp.get_context('fork')
    
    start_time = time.time()
    with ctx.Pool(processes=num_workers) as pool:
        # Menjalankan pemrosesan paralel
        results = pool.map(worker_task, chunks)
    end_time = time.time()
    
    return end_time - start_time

# --- 2. Eksekusi Eksperimen ---
if __name__ == '__main__':
    # Beban kerja: Mencari bilangan prima hingga 2.000.000 (Sesuaikan jika terlalu lama/cepat)
    N = 2000000
    data_input = np.arange(1, N)
    
    workers_list = [1, 2, 4, 8]
    execution_times = []

    print("Memulai eksperimen komputasi paralel...")
    for w in workers_list:
        t = run_experiment(w, data_input)
        execution_times.append(t)
        print(f"Worker = {w} | Waktu Eksekusi = {t:.4f} detik")

    # --- 3. Perhitungan Speedup Aktual & Teoritis (Hukum Amdahl) ---
    t1 = execution_times[0]  # Waktu eksekusi dengan 1 worker
    actual_speedup = [t1 / t for t in execution_times]

    # Asumsi fraksi paralel (P) pada Hukum Amdahl = 95% (0.95)
    # Sesuaikan nilai P ini dengan parameter teori dari Pertemuan 1 kalian
    P = 0.95 
    amdahl_speedup = [1 / ((1 - P) + (P / w)) for w in workers_list]

    # --- 4. Plotting Kurva ---
    plt.figure(figsize=(8, 5))
    plt.plot(workers_list, actual_speedup, marker='o', linestyle='-', color='b', label='Actual Speedup')
    plt.plot(workers_list, amdahl_speedup, marker='s', linestyle='--', color='r', label=f'Amdahl\'s Law (P={P})')
    plt.plot(workers_list, workers_list, linestyle=':', color='gray', label='Linear Speedup (Ideal)')

    plt.title('Analisis Speedup Skala vs Jumlah Worker')
    plt.xlabel('Jumlah Worker')
    plt.ylabel('Speedup')
    plt.xticks(workers_list)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()

In [ ]:
# Analisis:
# Eksperimen dilakukan untuk mengetahui pengaruh jumlah worker
# terhadap waktu eksekusi perhitungan bilangan prima.
# Pengujian dilakukan menggunakan 1, 2, 4, dan 8 worker.
#
# Secara teoretis, semakin banyak worker yang digunakan,
# semakin cepat pekerjaan yang dapat diproses secara paralel.
# Namun, hasil aktual dapat berbeda dari Hukum Amdahl karena
# jumlah core fisik yang terbatas, overhead pembuatan proses,
# serta beban kerja aplikasi lain yang berjalan bersamaan.
#
# Oleh karena itu, penambahan jumlah worker tidak selalu
# meningkatkan kecepatan secara proporsional. Performa aktual
# bergantung pada kemampuan perangkat keras dan overhead
# selama pemrosesan paralel.

In [ ]:
#nomor 2
import time
import random
import multiprocessing as mp
import sys

def cek_prima(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# 1. FUNGSI PRODUCER (Antar-Proses)
def producer(q, jumlah_consumers):
    print("[Producer] Mulai memproduksi data!")
    for i in range(1, 20): # Menghasilkan angka untuk diproses secara CPU-bound
        # Mengirimkan angka acak besar yang akan diuji primanya
        item = random.randint(100000, 500000)
        q.put(item)
        print(f"[Producer] Menambahkan angka: {item}")
        time.sleep(0.05) 
    
    print("[Producer] Selesai memproduksi semua data.")
    
    for _ in range(jumlah_consumers):
        q.put(None)

# 2. FUNGSI CONSUMER (Antar-Proses)
def consumer(consumer_id, q):
    print(f"  [Consumer-{consumer_id}] Mulai memproses data!")
    while True:
        item = q.get()
        
        # Cek sinyal berhenti (None)
        if item is None:
            print(f"  [Consumer-{consumer_id}] Menerima sinyal 'None', Berhenti.")
            break
        
        hasil = cek_prima(item)
        print(f"  [Consumer-{consumer_id}] Memproses angka {item} -> Apakah Prima? {hasil}")
        time.sleep(0.1) # Simulasi waktu proses komputasi

if __name__ == '__main__':
    # Memastikan kompatibilitas method context (aman untuk Windows & Linux)
    metode_ctx = 'spawn' if sys.platform.startswith('win') else 'fork'
    ctx = mp.get_context(metode_ctx)

    q = ctx.Queue()
    JUMLAH_CONSUMERS = 3

    # 3. MEMBUAT DAN MENJALANKAN PROSES
    p_producer = ctx.Process(target=producer, args=(q, JUMLAH_CONSUMERS), name="Producer-Process")

    p_consumers = []
    for i in range(1, JUMLAH_CONSUMERS + 1):
        p = ctx.Process(target=consumer, args=(i, q), name=f"Consumer-Process-{i}")
        p_consumers.append(p)

    start_time = time.time()

    p_producer.start()
    for p in p_consumers:
        p.start()

    # Tunggu semua proses selesai
    p_producer.join()
    for p in p_consumers:
        p.join()

    end_time = time.time()
    print(f"\n[Main] Semua proses Producer dan Consumer selesai dalam {end_time - start_time:.4f} detik!")


In [ ]:
1. Analisis Perbandingan
a. Versi Shared Memory (Value) umumnya mencatatkan waktu eksekusi yang sedikit lebih cepat. Hal ini karena proses langsung menuliskan data ke area memori bersama tanpa perlu melakukan serialisasi/deserialisasi pesan yang rumit.
b. Versi Message Passing (Queue) sedikit lebih lambat karena membutuhkan proses enkapsulasi data ke dalam antrean (queue) yang dikelola oleh sistem operasi, lalu dikirimkan dari proses anak kembali ke main process.
2. Aspek Keamanan Data & Sinkronisasi (Race Condition):
a. Pada Queue, sinkronisasi ditangani secara otomatis dan aman oleh struktur data antrean internal, sehingga risiko data rusak (race condition) sangat kecil karena setiap proses bekerja secara terisolasi sebelum mengirim data.
b. Pada Value, penggunaan objek Lock menjadi kunci mutlak. Tanpa adanya Lock, beberapa proses yang mencoba mengubah nilai variabel bersamaan akan saling menimpa data satu sama lain (race condition), yang berakibat pada hasil total akhir yang salah. Namun, terlalu sering menggunakan Lock juga bisa memicu lock contention (proses saling menunggu giliran membuka kunci), yang dapat memperlambat kinerja jika tidak dikelola dengan baik.
Mengenai Kode yang tidak bisa di run
Di Windows, mekanisme spawn mengharuskan Python meluncurkan sub-process baru dari awal dan melakukan pickling terhadap fungsi targetnya. Sering kali, VSCode gagal menangani stream input/output antarproses tersebut, sehingga program terjebak dalam deadlock permanen tanpa memunculkan error. Karena itu tidak bisa di run atau tidak berhenti berhenti prosesnya saat dijalankan.

In [ ]:
import time
import random
import multiprocessing as mp
import sys

def cek_prima(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# 1. FUNGSI PRODUCER (Antar-Proses)
def producer(q, jumlah_consumers):
    print("[Producer] Mulai memproduksi data!")
    for i in range(1, 20): # Menghasilkan angka untuk diproses secara CPU-bound
        # Mengirimkan angka acak besar yang akan diuji primanya
        item = random.randint(100000, 500000)
        q.put(item)
        print(f"[Producer] Menambahkan angka: {item}")
        time.sleep(0.05) 
    
    print("[Producer] Selesai memproduksi semua data.")
    
    for _ in range(jumlah_consumers):
        q.put(None)

# 2. FUNGSI CONSUMER (Antar-Proses)
def consumer(consumer_id, q):
    print(f"  [Consumer-{consumer_id}] Mulai memproses data!")
    while True:
        item = q.get()
        
        # Cek sinyal berhenti (None)
        if item is None:
            print(f"  [Consumer-{consumer_id}] Menerima sinyal 'None', Berhenti.")
            break
        
        hasil = cek_prima(item)
        print(f"  [Consumer-{consumer_id}] Memproses angka {item} -> Apakah Prima? {hasil}")
        time.sleep(0.1) # Simulasi waktu proses komputasi

if __name__ == '__main__':
    # Memastikan kompatibilitas method context (aman untuk Windows & Linux)
    metode_ctx = 'spawn' if sys.platform.startswith('win') else 'fork'
    ctx = mp.get_context(metode_ctx)

    q = ctx.Queue()
    JUMLAH_CONSUMERS = 3

    # 3. MEMBUAT DAN MENJALANKAN PROSES
    p_producer = ctx.Process(target=producer, args=(q, JUMLAH_CONSUMERS), name="Producer-Process")

    p_consumers = []
    for i in range(1, JUMLAH_CONSUMERS + 1):
        p = ctx.Process(target=consumer, args=(i, q), name=f"Consumer-Process-{i}")
        p_consumers.append(p)

    start_time = time.time()

    p_producer.start()
    for p in p_consumers:
        p.start()

    # Tunggu semua proses selesai
    p_producer.join()
    for p in p_consumers:
        p.join()

    end_time = time.time()
    print(f"\n[Main] Semua proses Producer dan Consumer selesai dalam {end_time - start_time:.4f} detik!")

In [ ]:
Nomor 3.
1. Analisis Aktual Perbandingan Versi Threading vs Multiprocessing (Producer-Consumer)
a. Perubahan Struktur Kode:
Secara logika arsitektur, pola kode tidak banyak berubah karena kedua versi sama-sama menggunakan konsep antrean (Queue) untuk memasukkan data (put) dan mengambil data (get).
Perbedaan struktural terletak pada pustaka yang diimpor (dari threading dan queue menjadi multiprocessing dan mp.Queue), penggunaan objek proses (mp.Process), serta keharusan menambahkan blok pengaman if name == 'main': yang wajib ada pada program multiprocessing di Python.
b. Perbandingan Performa untuk Tugas CPU-bound:
Versi Threading (Pertemuan 2): Kinerja akan sangat lambat atau mengalami bottleneck yang parah jika diberikan tugas komputasi berat (CPU-bound seperti pengecekan bilangan prima). Hal ini disebabkan oleh adanya aturan GIL (Global Interpreter Lock) pada Python, di mana dalam satu waktu, interpreter hanya mengizinkan satu thread yang mengeksekusi bytecode Python murni pada satu inti CPU fisik. Akibatnya, meskipun dibuat banyak thread, beban kerja tetap bergantian dikerjakan pada satu core saja.
Versi Multiprocessing: Karena setiap proses memiliki interpreter dan ruang memori yang independen secara fisik, pembatasan GIL berhasil dilewati sepenuhnya (true parallelism). Program dapat mendistribusikan beban kerja komputasi berat ke berbagai inti (core) CPU secara simultan. Meskipun ada sedikit waktu tambahan untuk overhead pembuatan proses (spawn overhead), skalabilitas eksekusi untuk tugas CPU-bound jauh lebih optimal dan efisien dibandingkan menggunakan thread.
2. Analisa tentang output
Ketika kode multiprocessing dijalankan di lingkungan lokal Windows (seperti di VS Code), program mengeksekusi metode spawn sebagai pengganti metode fork. Pada mekanisme spawn, setiap proses anak (child process) diluncurkan sebagai interpreter Python yang benar-benar terisolasi dengan ruang memori yang terpisah dari proses utama (main process).
Penyebab log dari fungsi producer dan consumer tidak muncul di terminal utama meliputi:
Buffer Standar Input/Output (stdout): Pada banyak lingkungan eksekusi terminal di Windows, output yang dihasilkan oleh proses anak (child process) masuk ke dalam penampungan memori (buffer) sistem operasi dan tidak langsung dialirkan (flushed) ke layar terminal utama kecuali proses tersebut selesai atau buffer penuh.
Keterlambatan / Kegagalan Aliran Stream: Karena proses berjalan sangat cepat (hanya sepersekian detik), program utama sudah keburu selesai dan menutup seluruh saluran komunikasi sebelum proses anak sempat mencetak teks ke layar konsol terminal interaktif.

In [ ]:
Nomor 4.
1. Kapan memilih Manager():
Dipilih untuk skenario kompleks yang membutuhkan struktur data fleksibel, seperti dictionary atau list dinamis (misalnya: aplikasi web scraper multiproses yang mengumpulkan metadata produk secara bersamaan ke dalam satu dict terpusat). Trade-off: Menawarkan kemudahan pengembangan karena mendukung Python object native, tetapi memiliki performa lebih lambat (overhead komunikasi antar-proses via IPC/server-process).

2. Kapan memilih Value / Array:
Dipilih untuk skenario high-performance computational dengan tipe data primitif dan ukuran tetap (misalnya: pemrosesan matriks gambar/piksel atau perhitungan numerik skala besar secara paralel). Trade-off: Menawarkan performa & kecepatan tinggi (karena berbagi shared memory langsung di RAM), namun kurang fleksibel karena tipe data terbatas dan ukurannya fixed/kaku.